# Join fan-out: aggregate before you join

Practice notebook for the datafromzero SQL lesson **Join fan-out: aggregate before you join** (S21). The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Build `order_totals`

In [ ]:
con.sql("""
WITH order_totals AS (
  SELECT oi.order_id,
         SUM(oi.quantity * p.unit_price_usd - oi.discount_usd) AS order_total_usd
  FROM order_items AS oi
  INNER JOIN products AS p ON oi.product_id = p.product_id
  GROUP BY oi.order_id
)
SELECT COUNT(*) AS orders,
       COUNT(DISTINCT order_id) AS distinct_ids,
       SUM(order_total_usd) AS total_usd
FROM order_totals
""").show()

### Step 2. Join the totals to the lines

In [ ]:
con.sql("""
WITH order_totals AS (
  SELECT oi.order_id,
         SUM(oi.quantity * p.unit_price_usd - oi.discount_usd) AS order_total_usd
  FROM order_items AS oi
  INNER JOIN products AS p ON oi.product_id = p.product_id
  GROUP BY oi.order_id
)
SELECT COUNT(*) AS joined_rows,
       SUM(t.order_total_usd) AS total_usd
FROM order_totals AS t
INNER JOIN order_items AS oi ON t.order_id = oi.order_id
""").show()

### Step 3. Ask the question the naive way

In [ ]:
con.sql("""
WITH order_totals AS (
  SELECT oi.order_id,
         SUM(oi.quantity * p.unit_price_usd - oi.discount_usd) AS order_total_usd
  FROM order_items AS oi
  INNER JOIN products AS p ON oi.product_id = p.product_id
  GROUP BY oi.order_id
),
lines AS (
  SELECT oi.order_item_id, oi.order_id, p.category = 'coffee' AS is_coffee
  FROM order_items AS oi
  INNER JOIN products AS p ON oi.product_id = p.product_id
)
SELECT l.is_coffee,
       COUNT(DISTINCT t.order_id) AS orders,
       SUM(t.order_total_usd) AS revenue_usd
FROM order_totals AS t
INNER JOIN lines AS l ON t.order_id = l.order_id
GROUP BY l.is_coffee
ORDER BY l.is_coffee DESC
""").show()

### Step 4. See the copies

In [ ]:
con.sql("""
WITH order_totals AS (
  SELECT oi.order_id,
         SUM(oi.quantity * p.unit_price_usd - oi.discount_usd) AS order_total_usd
  FROM order_items AS oi
  INNER JOIN products AS p ON oi.product_id = p.product_id
  GROUP BY oi.order_id
),
lines AS (
  SELECT oi.order_item_id, oi.order_id, p.category = 'coffee' AS is_coffee
  FROM order_items AS oi
  INNER JOIN products AS p ON oi.product_id = p.product_id
)
SELECT l.order_item_id, l.order_id, l.is_coffee, t.order_total_usd
FROM order_totals AS t
INNER JOIN lines AS l ON t.order_id = l.order_id
WHERE t.order_id = 1001
ORDER BY l.order_item_id
""").show()

### Step 5. Repair: reduce the many side to one row per order

In [ ]:
con.sql("""
WITH order_flags AS (
  SELECT oi.order_id,
         MAX(CASE WHEN p.category = 'coffee' THEN 1 ELSE 0 END) AS has_coffee
  FROM order_items AS oi
  INNER JOIN products AS p ON oi.product_id = p.product_id
  GROUP BY oi.order_id
)
SELECT COUNT(*) AS orders,
       COUNT(DISTINCT order_id) AS distinct_ids,
       SUM(has_coffee) AS coffee_orders
FROM order_flags
""").show()

### Step 6. Join two tables with one row per order

In [ ]:
con.sql("""
WITH order_totals AS (
  SELECT oi.order_id,
         SUM(oi.quantity * p.unit_price_usd - oi.discount_usd) AS order_total_usd
  FROM order_items AS oi
  INNER JOIN products AS p ON oi.product_id = p.product_id
  GROUP BY oi.order_id
),
order_flags AS (
  SELECT oi.order_id,
         MAX(CASE WHEN p.category = 'coffee' THEN 1 ELSE 0 END) AS has_coffee
  FROM order_items AS oi
  INNER JOIN products AS p ON oi.product_id = p.product_id
  GROUP BY oi.order_id
)
SELECT f.has_coffee,
       COUNT(*) AS orders,
       SUM(t.order_total_usd) AS revenue_usd
FROM order_totals AS t
INNER JOIN order_flags AS f ON t.order_id = f.order_id
GROUP BY f.has_coffee
ORDER BY f.has_coffee DESC
""").show()

### Step 7. Reconcile

In [ ]:
con.sql("""
SELECT COUNT(DISTINCT oi.order_id) AS coffee_orders
FROM order_items AS oi
INNER JOIN products AS p ON oi.product_id = p.product_id
WHERE p.category = 'coffee'
""").show()

## Exercise 1. Orders that contained tea

How much did orders that contained tea bring in, and how much did the others? Given: `order_totals` from step 1, and the step that gives one row per order. The cell below runs as it is and shows that step. Replace its last line with the query that joins the two on `order_id`, groups by `has_tea` and shows the number of orders and the revenue. The two revenues should add up to step 1's control total.

In [ ]:
con.sql("""
WITH order_totals AS (
  SELECT oi.order_id,
         SUM(oi.quantity * p.unit_price_usd - oi.discount_usd) AS order_total_usd
  FROM order_items AS oi
  INNER JOIN products AS p ON oi.product_id = p.product_id
  GROUP BY oi.order_id
),
order_flags AS (
  SELECT oi.order_id,
         MAX(CASE WHEN p.category = 'tea' THEN 1 ELSE 0 END) AS has_tea
  FROM order_items AS oi
  INNER JOIN products AS p ON oi.product_id = p.product_id
  GROUP BY oi.order_id
)
SELECT * FROM order_flags LIMIT 5  -- replace with your query
""").show()

## Exercise 2. Revenue by payment method

How much did each payment method bring in, and in how many orders? A teammate says the three amounts must add up to E21's Sales, 22,749.25. Before you run anything, write two sentences in the cell below: the grain of `orders` and of `order_totals`, and whether joining them can multiply rows, with the number of rows you expect. Then write the query, and compare its total with both numbers. The `order_totals` step is given in the cell after the sentences.

In [ ]:
# My expected result, in words:
#

In [ ]:
con.sql("""
WITH order_totals AS (
  SELECT oi.order_id,
         SUM(oi.quantity * p.unit_price_usd - oi.discount_usd) AS order_total_usd
  FROM order_items AS oi
  INNER JOIN products AS p ON oi.product_id = p.product_id
  GROUP BY oi.order_id
)
SELECT * FROM order_totals LIMIT 5  -- replace with your query
""").show()

## Exercise 3. Spot the flaw

A colleague wants the average order value and reuses the join from step 2. Run it, then find out what went wrong and write the right query.

In [ ]:
con.sql("""
WITH order_totals AS (
  SELECT oi.order_id,
         SUM(oi.quantity * p.unit_price_usd - oi.discount_usd) AS order_total_usd
  FROM order_items AS oi
  INNER JOIN products AS p ON oi.product_id = p.product_id
  GROUP BY oi.order_id
)
SELECT ROUND(AVG(t.order_total_usd), 2) AS avg_order_usd
FROM order_totals AS t
INNER JOIN order_items AS oi ON t.order_id = oi.order_id
""").show()

In [ ]:
# Your corrected query: